# Terra-Style Jupyter Demo

This notebook is part of the Heartwood open-source project. Copyright and license metadata are declared in `REUSE.toml`.

This notebook exercises the synthetic Heartwood reference analysis in a Terra-style Jupyter workspace. It uses the same persisted session as the CLI and researcher web UI, keeps all data synthetic, and does not claim real Terra identity binding, BigQuery access, controlled-data validation, or institutional approval. Use the no-weight Terra-derived `0.1.0-terra` release image.


## Terminal Setup

Use `ghcr.io/schmiedmayerlab/heartwood:0.1.0-terra`. The image contains no model weights. Configure either an institution-authorized provider profile or an OpenAI-compatible local service as described in `docs/terra-jupyter-demo.md`, then complete the analysis and audit cells below. Do not submit notebook commands while a separate web gateway is active for the same session. After the notebook task is idle, start the web interface from a Jupyter terminal:

```bash
cd /opt/heartwood
export HEARTWOOD_WORKSPACE=/home/jupyter/heartwood-workspace/sessions
export HEARTWOOD_WEB_HOST=127.0.0.1
export HEARTWOOD_WEB_PORT=8767
bash images/generic/scripts/start_web_ui.sh
```

Open the notebook proxy URL for port `8767`. In Terra-style `jupyter-server-proxy` routes, the browser usually sees `/user/<name>/proxy/8767/` while the proxy strips that prefix before forwarding to Heartwood. The pull-request integration uses a deterministic loopback fixture through the real OpenHands SDK and makes no model-quality claim.


In [ ]:
import json
from pathlib import Path
from shutil import copy2

from heartwood.notebook import NotebookSession, build_widget_spec, jupyter_proxy_url

state_root = Path.home() / "heartwood-workspace"
workspace = state_root / "sessions"
session_id = "terra-demo"
analysis_workspace = state_root / "workspaces" / session_id
input_root = analysis_workspace / "input"
fixture_root = Path("/opt/heartwood/fixtures/synthetic/omop-like")
input_root.mkdir(parents=True, exist_ok=True)
for filename in ("person.csv", "condition_occurrence.csv"):
    copy2(fixture_root / filename, input_root / filename)

session = NotebookSession(workspace=workspace, session_id=session_id)
print("Web UI proxy URL:", jupyter_proxy_url(port=8767))
print("Session workspace:", analysis_workspace)
print("Localized inputs:", sorted(path.name for path in input_root.iterdir()))

In [ ]:
detection = session.detect()

print("Dataset proposals")
for proposal in detection.dataset_proposals:
    print(f"- {proposal.dataset_type} ({proposal.confidence:.2f})")
    for item in proposal.evidence:
        print(f"  - {item}")

print("Latest activity:", detection.activity[-1])

In [ ]:
prompt = (
    "Build the synthetic target-condition cohort for concept 201826 with the "
    "repository-verified cohort Skill. Read the localized tables in input, require "
    "age 18 or older, apply an aggregate count floor of 20, write "
    "cohort-summary.json, and report quality checks without row-level values."
)
run = session.run(prompt)

print("Policy decisions")
for policy in run.policy_status:
    print(f"- {policy.decision}: {policy.endpoint} ({policy.reason})")

print("Approval controls")
for control in run.approval_controls:
    print(f"- {control.target_type}: {control.target_id} [{control.decision or 'pending'}]")

print("Activity")
for item in run.activity:
    print(f"{item.sequence:03d} {item.kind}: {item.detail}")

## Review And Allow The Synthetic Action

Review the printed pending action in the web conversation or CLI before running the next cell. Continue only when it invokes the repository-verified cohort script, reads `input`, writes `cohort-summary.json` inside this analysis workspace, and does not transmit data.


In [ ]:
pending = [control for control in run.approval_controls if control.decision is None]
if len(pending) != 1:
    raise RuntimeError(f"Expected one pending synthetic action, found {len(pending)}")
completed = session.approve(tool_call_id=pending[0].target_id)

cohort_path = analysis_workspace / "cohort-summary.json"
cohort = json.loads(cohort_path.read_text(encoding="utf-8"))
assert cohort["summary"]["source_participant_count"] == 24
assert cohort["summary"]["source_condition_occurrence_count"] == 39
assert cohort["summary"]["participant_count"] == 20
assert cohort["summary"]["condition_occurrence_count"] == 35
assert cohort["quality_checks"]["row_values_exported"] is False
print(json.dumps(cohort, indent=2, sort_keys=True))

In [ ]:
exported = session.audit_export()
replayed = session.replay()

print("Persisted events:", replayed.event_count)
print("Exports")
for action in exported.export_actions:
    print(f"- {action.label}: {action.path}")

print("Notebook widget sections:", [section.title for section in build_widget_spec(replayed)])

## CLI And Web Replay Evidence

The CLI and web interface read the same persisted events. Open the `terra-demo` session in the web UI, then run this command from a Jupyter terminal:

```bash
heartwood --workspace /home/jupyter/heartwood-workspace/sessions --session-id terra-demo replay
```

The notebook, CLI, and web UI should show the same researcher task, route decision, proposed terminal action, approval, successful tool outcome, and final model response. A live Terra validation pass still needs to confirm immutable-image launch, Leonardo proxy behavior, inherited identity, persistent storage, autopause and resume, and deployment-specific data and model controls.
